# NPS Agent with MLflow Tracing & Agent-as-a-Judge

Query the National Parks Service using LlamaStack + MCP, with MLflow tracing and automated evaluation.

**Prerequisites:**
- LlamaStack server on `localhost:8321`
- NPS MCP server on `localhost:3005`
- `OPENAI_API_KEY` in environment

In [9]:
import os
import time
from dotenv import load_dotenv

# Load .env from parent directory (agents_tracing-eval_mlflow/.env)
env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

import mlflow
from mlflow.entities import SpanType, AssessmentSource, AssessmentSourceType
from mlflow.genai.judges import make_judge
from llama_stack_client import LlamaStackClient
from typing import Literal

## Configuration

In [10]:
# Configuration
LLAMA_STACK_URL = "http://localhost:8321/"
# Use host.containers.internal so OGX (running in Podman) can reach the MCP server on the host
NPS_MCP_URL = "http://host.containers.internal:3005/sse/"
MODEL_ID = "openai/gpt-4o"
JUDGE_MODEL = "openai:/gpt-4o"

In [11]:

db_path = os.path.join(os.getcwd(), "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("nps-agent")
print(f"MLflow database: {db_path}")

MLflow database: /Users/sschifma/DEV/rh_repos/agents/examples/agents_tracing-eval_mlflow/nps_agent/mlflow.db


## Agent Function

Queries NPS via LlamaStack with MCP tools attached. The `@mlflow.trace` decorator captures the execution.

In [12]:
@mlflow.trace(name="query_nps", span_type=SpanType.AGENT)
def query_nps(prompt: str, model: str = MODEL_ID) -> str:
    """Query the National Parks Service agent."""
    client = LlamaStackClient(base_url=LLAMA_STACK_URL)
    
    with mlflow.start_span(name="mcp_tool_call", span_type=SpanType.LLM) as span:
        span.set_inputs({"model": model, "prompt": prompt})

        start_time = time.time()
        response = client.responses.create(
            model=model,
            input=prompt,
            tools=[{"type": "mcp", "server_url": NPS_MCP_URL, "server_label": "NPS tools"}]
        )
        latency_ms = (time.time() - start_time) * 1000

        token_metrics = {}
        if response.usage:
            token_metrics = {
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "total_tokens": response.usage.total_tokens,
            }

        mlflow.log_metrics({"latency_ms": latency_ms, **token_metrics})

        span.set_outputs({
            "response_id": response.id,
            "status": response.status,
            "latency_ms": latency_ms,
            **token_metrics,
        })
    
    # Extract text response
    for output in response.output:
        if output.type in ("text", "message") and hasattr(output, 'content') and output.content:
            return output.content[0].text
    return ""

## Agent-as-a-Judge

An Agent that evaluates the agent's trace after execution. Instead of just looking at inputs/outputs, it uses tools to inspect the full execution:
- What spans were created
- What tools were called
- How long each step took

The `{{ trace }}` in the instructions tells MLflow to give the judge these inspection tools.

In [13]:
# Agent-as-a-Judge scorer
nps_judge = make_judge(
    name="nps_agent_evaluator",
    instructions=(
        "Evaluate the NPS agent's performance in {{ trace }}.\n\n"
        "Check for:\n"
        "1. Response Quality: Did the agent correctly identify parks and provide accurate information?\n"
        "2. Tool Usage: Were the correct NPS MCP tools used (search_parks, get_park_events, etc.)?\n"
        "3. Completeness: Did the agent answer all parts of the user's question?\n\n"
        "Rate as: 'good', 'acceptable', or 'poor'"
    ),
    feedback_value_type=Literal["good", "acceptable", "poor"],
    model=JUDGE_MODEL,
)

In [14]:
def evaluate_trace(trace):
    """Run Agent-as-a-Judge evaluation and log to MLflow."""
    feedback = nps_judge(trace=trace)
    
    trace_id = trace.info.trace_id
    mlflow.log_feedback(
        trace_id=trace_id,
        name="nps_agent_evaluation",
        value=feedback.value,
        rationale=feedback.rationale,
        source=AssessmentSource(
            source_type=AssessmentSourceType.LLM_JUDGE,
            source_id=f"agent-as-a-judge/{JUDGE_MODEL}",
        ),
    )
    
    print(f"\nEvaluation: {feedback.value}")
    print(f"Rationale: {feedback.rationale}")
    return feedback


## Run Agent & Evaluate

1. Send a query to the NPS agent
2. Get the MLflow trace from the execution
3. Pass the trace to the judge for evaluation
4. Log the feedback to MLflow (visible in Assessments panel)

In [15]:
prompt = "Tell me about some parks in Rhode Island, and let me know if there are any upcoming events at them."

with mlflow.start_run(run_name=f"nps-{MODEL_ID}"):
    result = query_nps(prompt)
    print(f"Response:\n{result}")

    # Flush async trace writes before retrieving
    mlflow.flush_trace_async_logging()

    # Evaluate the trace
    trace_id = mlflow.get_last_active_trace_id()
    trace = mlflow.get_trace(trace_id)
    evaluate_trace(trace)

Response:
Here's an overview of some parks in Rhode Island along with their upcoming events:

### 1. Blackstone River Valley National Historical Park
The Blackstone River powered America's entry into the Age of Industry. This park commemorates the industrial heritage and landmarks within the Blackstone Valley.

#### Upcoming Events:
- **Old Slater Mill Tour**: Learn about the American Industrial Revolution with a ranger-led tour. Tours start at 10:30 AM, 12:30 PM, and 2:30 PM.
- **First Strike Commemoration**: Participate in hands-on activities and a weaving workshop focused on the wage workers' strike of 1824.
- **Take Me Fishing**: Free family fishing activities available from 11:00 AM - 4:00 PM on Sundays.
- **Nature Walk and Scavenger Hunt**: Join a nature walk and scavenger hunt at Blackstone River State Park.
- Many more events including ranger walkabouts and educational programs.

More details can be found on the [Blackstone River Valley NHP website](https://www.nps.gov/blrv/ind

## View Traces in MLflow UI

Start the MLflow UI to view traces and assessments:

```bash
mlflow ui --port 5001
```

Then open http://localhost:5001 in your browser.

### How to Navigate

1. **Select the Experiment** - Click on `nps-agent` in the left sidebar
2. **Go to Traces tab** - Click the "Traces" tab to see all agent executions
3. **View Trace Details** - Click on any Trace ID to open the trace detail view
   - You'll see the span hierarchy showing the agent execution (query_nps → mcp_tool_call)
   - Click on individual spans to see inputs/outputs for each step
4. **View Assessments** - In the trace detail view, look for the assessments side-panel on the right
   - This shows the Agent-as-a-Judge evaluation results.